# 00 - Environment and Sanity Checks (DINOv2/DINOv3-ready)

This notebook prepares a reliable environment for the rest of the training.

## Goals
- Create and validate Conda + Python runtime
- Install core CV packages
- Download/cache lightweight datasets and test assets
- Load DINOv2 now, with a DINOv3-ready loading interface


## 1) Conda environment setup (run in terminal)

### Why two steps instead of only `pip install -r requirements.txt`?

- **`requirements.txt` does not pin PyTorch** on purpose. A single file that lists `torch` without PyTorch’s **wheel index** often installs **CPU-only** builds from PyPI, or the wrong CUDA variant — so an RTX 8000 might never use the GPU.
- **Fix:** install **torch / torchvision / torchaudio** in one explicit command (GPU or CPU index), then `pip install -r requirements.txt` for everything else.

### Commands (from repo root)

```bash
conda create -n dino-fasttrack python=3.10 -y
conda activate dino-fasttrack
pip install --upgrade pip

# Pick ONE: GPU (CUDA 12.1 wheels) — best for RTX 8000 if drivers are OK
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# OR CPU-only fallback (slower; notebooks still work)
# pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

pip install -r requirements.txt
python -m ipykernel install --user --name dino-fasttrack --display-name "Python (dino-fasttrack)"
```

**Git Bash:** if `conda activate` says *Run `conda init` first*, run `conda init bash`, **restart** Git Bash, then activate again. Or use PowerShell/cmd, or `conda run -n dino-fasttrack ...` (see README).

**“No CUDA installed”:** you usually **do not** install the full CUDA Toolkit by hand. GPU PyTorch wheels (`cu121`) bundle the CUDA **runtime**; you need an **NVIDIA driver** so `nvidia-smi` works. No GPU / no driver → use the **CPU** `pip install` line above.

**Drivers:** update NVIDIA drivers if `nvidia-smi` fails or `torch.cuda.is_available()` is False after GPU install. If GPU still won’t work, use the CPU `pip install` line above — code in these notebooks uses `cuda if available else cpu`.

See also **`README.md`** for the same flow and Conda ToS troubleshooting.


In [ ]:
import os
import platform
import torch

print('Python platform:', platform.platform())
print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## 2) Workspace paths
This cell creates folders used by all notebooks.

In [ ]:
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_DIR = ROOT / 'data'
ARTIFACTS_DIR = ROOT / 'artifacts'
WEIGHTS_DIR = ROOT / 'weights'

for p in [DATA_DIR, ARTIFACTS_DIR, WEIGHTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)
print('DATA_DIR:', DATA_DIR)
print('ARTIFACTS_DIR:', ARTIFACTS_DIR)
print('WEIGHTS_DIR:', WEIGHTS_DIR)


## 3) Quick dataset bootstrap (small and flight-friendly)

We use CIFAR-10 for quick classification experiments and keep everything lightweight.
For anomaly and segmentation datasets, we provide download commands in later notebooks.


In [ ]:
from torchvision import datasets

cifar_train = datasets.CIFAR10(root=str(DATA_DIR), train=True, download=True)
cifar_test = datasets.CIFAR10(root=str(DATA_DIR), train=False, download=True)
print('CIFAR-10 train size:', len(cifar_train))
print('CIFAR-10 test size:', len(cifar_test))


## 4) DINO model loader abstraction

This function prioritizes DINOv3 if a compatible model name exists in your environment.
If not, it falls back to DINOv2 via timm.


In [ ]:
import timm

def load_dino_backbone(prefer_v3=True, device=None):
    if device is None:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'

    candidate_models = []
    if prefer_v3:
        candidate_models += [
            'dinov3_vitb16',
            'dinov3_vitl16',
        ]
    candidate_models += [
        'vit_base_patch14_dinov2.lvd142m',
        'vit_small_patch14_dinov2.lvd142m',
    ]

    available = set(timm.list_models(pretrained=True))
    chosen = None
    for name in candidate_models:
        if name in available:
            chosen = name
            break

    if chosen is None:
        raise RuntimeError('No matching DINO model found in timm pretrained registry.')

    model = timm.create_model(chosen, pretrained=True, num_classes=0)
    model.eval().to(device)
    return model, chosen, device

model, model_name, device = load_dino_backbone(prefer_v3=True)
print('Loaded model:', model_name)
print('Device:', device)


## 5) Single-image embedding sanity test
This verifies end-to-end preprocessing and feature extraction.

In [ ]:
from torchvision import transforms
from PIL import Image
import requests
from io import BytesIO

url = 'https://images.pexels.com/photos/404280/pexels-photo-404280.jpeg'
img = Image.open(BytesIO(requests.get(url, timeout=30).content)).convert('RGB')

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

x = transform(img).unsqueeze(0).to(device)
with torch.no_grad():
    emb = model(x)

print('Embedding shape:', tuple(emb.shape))


## 6) What to do next

1. Confirm all cells pass.
2. Keep this kernel selected for all remaining notebooks.
3. Move to `01_dinov2_image_classification_and_embeddings.ipynb`.
